# Drift Analysis — Credit Scoring API

## Objectif
Détecter si les **données de production** ont dérivé par rapport aux données d'entraînement.  
Une dérive significative signifie que le modèle reçoit en production des données différentes de celles sur lesquelles il a été entraîné — ses prédictions peuvent donc se dégrader.

## Approche — simulation de la production par le temps
Le dataset Home Credit ne dispose pas de données de production réelles.  
On simule l'écoulement du temps via `SK_ID_CURR` : un identifiant plus élevé = une demande plus récente.

| Jeu de données | Contenu | Rôle |
|---|---|---|
| **Référence** | 80% des clients les plus anciens | Ce que le modèle a vu à l'entraînement |
| **Production simulée** | 20% des clients les plus récents | Ce que le modèle reçoit en production |

## Outil : Evidently AI
Evidently compare les distributions des features entre les deux jeux et identifie celles qui ont dérivé.

## 1. Imports

In [ ]:
import re
from pathlib import Path

import joblib
import pandas as pd
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

## 2. Chargement du dataset

In [ ]:
PARQUET_PATH = Path("../data/processed/train_processed_global.parquet")
CSV_PATH     = Path("../data/processed/train_processed_global.csv")

# Parquet en priorité (167MB vs 733MB pour le CSV)
if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    print(f"Chargé depuis Parquet : {PARQUET_PATH.name}")
elif CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    print(f"Chargé depuis CSV : {CSV_PATH.name}")
else:
    raise FileNotFoundError("Dataset introuvable — vérifiez data/processed/")

# Nettoyage des noms de colonnes — même transformation que dans l'API
# LightGBM rejette les caractères spéciaux ([, ], <, espace...)
df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]

print(f"Dataset : {df.shape[0]:,} clients × {df.shape[1]} colonnes")
df.head(3)

## 3. Split référence / production simulée

`SK_ID_CURR` est séquentiel : les identifiants les plus élevés correspondent aux demandes les plus récentes.  
On trie par cet identifiant pour simuler l'écoulement du temps.

In [ ]:
# Tri par SK_ID_CURR croissant (ancien → récent)
df_sorted = df.sort_values("SK_ID_CURR").reset_index(drop=True)

split_idx = int(len(df_sorted) * 0.8)

reference = df_sorted.iloc[:split_idx]   # 80% les plus anciens
current   = df_sorted.iloc[split_idx:]   # 20% les plus récents

print(f"Référence  : {len(reference):,} clients — SK_ID_CURR jusqu'à {reference['SK_ID_CURR'].max():,}")
print(f"Production : {len(current):,}  clients — SK_ID_CURR à partir de {current['SK_ID_CURR'].min():,}")

## 4. Distribution de la cible (TARGET)

Vérification préliminaire : le taux de défaut est-il stable entre les deux périodes ?

In [ ]:
print("Taux de défaut (TARGET = 1) :")
print(f"  Référence  : {reference['TARGET'].mean():.2%} ({reference['TARGET'].sum():,} défauts sur {len(reference):,})")
print(f"  Production : {current['TARGET'].mean():.2%} ({current['TARGET'].sum():,} défauts sur {len(current):,})")

delta = current['TARGET'].mean() - reference['TARGET'].mean()
print(f"\n  Écart : {delta:+.2%}")

## 5. Sélection des features du modèle

On restreint l'analyse aux features utilisées par LightGBM (552 colonnes) —  
ce sont celles qui influencent directement les prédictions.

In [ ]:
MODEL_PATH = Path("../models/lgbm_optimized.pkl")
model      = joblib.load(MODEL_PATH)

# feature_name_ : liste des colonnes dans l'ordre attendu par le modèle
feature_cols = [c for c in model.feature_name_ if c in df.columns]

print(f"Features du modèle trouvées dans le dataset : {len(feature_cols)} / {len(model.feature_name_)}")

ref_features = reference[feature_cols]
cur_features = current[feature_cols]

print(f"Référence  : {ref_features.shape}")
print(f"Production : {cur_features.shape}")

## 6. Rapport de drift Evidently

`DataDriftPreset` teste chaque feature avec un test statistique adapté à son type :
- Variables numériques → test de Wasserstein ou Kolmogorov-Smirnov
- Variables catégorielles → test du Chi²

Une feature est considérée comme **dérivée** si la p-value est inférieure au seuil (0.05 par défaut).

In [ ]:
# Définition du schéma de données (pas de cible ici — analyse des features uniquement)
data_def = DataDefinition()

ref_ds = Dataset.from_pandas(ref_features, data_definition=data_def)
cur_ds = Dataset.from_pandas(cur_features, data_definition=data_def)

# Rapport de drift — peut prendre quelques minutes sur 552 features
report   = Report([DataDriftPreset()])
snapshot = report.run(reference_data=ref_ds, current_data=cur_ds)

print("Rapport généré.")

## 7. Sauvegarde du rapport HTML

In [ ]:
REPORT_PATH = Path("../reports/drift_report.html")
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

snapshot.save_html(str(REPORT_PATH))
print(f"Rapport sauvegardé : {REPORT_PATH.resolve()}")
print("Ouvrir ce fichier dans un navigateur pour visualiser le rapport interactif.")

## 8. Résumé des résultats

Aperçu rapide dans le notebook avant d'ouvrir le rapport complet.

In [ ]:
# Extraction du résultat de drift global depuis le snapshot
results = snapshot.dict()

# Recherche du résultat DatasetDriftMetric dans les métriques
for metric in results.get("metrics", []):
    name = metric.get("metric", "")
    if "DatasetDrift" in str(name):
        print(f"Drift détecté : {metric.get('value', {}).get('dataset_drift', 'N/A')}")
        print(f"Features dérivées : {metric.get('value', {}).get('number_of_drifted_columns', 'N/A')} / {len(feature_cols)}")
        share = metric.get('value', {}).get('share_of_drifted_columns', None)
        if share is not None:
            print(f"Part des features dérivées : {share:.1%}")
        break

print("\n→ Ouvrir reports/drift_report.html pour le rapport complet interactif.")

## Conclusion

### Interprétation

- Si **peu de features dérivent** (< 20%) : les données de production sont stables, le modèle reste fiable.
- Si **beaucoup de features dérivent** (> 30%) : la distribution des données a changé — envisager un réentraînement.

### Limites de cette analyse

- La simulation par `SK_ID_CURR` est un **proxy temporel**, pas une vraie temporalité (les dates exactes des demandes ne sont pas dans le dataset).
- En production réelle, la référence serait les données d'entraînement et le *current* serait le flux de requêtes loggées par l'API.

### Prochaine étape

Brancher Evidently sur les **logs JSON de l'API** (générés à chaque appel `/predict`) pour monitorer le drift en continu sur de vraies données de production.